# Differential Equations — Session 12
## Section 3.3: Modeling with Systems of First-Order Differential Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. distinguish linear and nonlinear systems;
2. formulate compartment systems using balance laws;
3. model a radioactive decay chain;
4. model solute exchange between connected tanks;
5. formulate and interpret a Lotka–Volterra predator–prey system;
6. identify equilibria of a two-species system;
7. interpret time series and phase-plane trajectories;
8. formulate a competition model and compare possible outcomes.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–15 min | Systems, vector form, and balance laws |
| 15–30 min | Radioactive decay chain |
| 30–48 min | Coupled mixing tanks |
| 48–74 min | Predator–prey model |
| 74–87 min | Competition model |
| 87–90 min | Exit check |

Electrical networks are retained as a short extension because their detailed solution methods appear later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def safe_solve(rhs, t_span, y0, points=700, **kwargs):
    t_eval = np.linspace(t_span[0], t_span[1], points)
    return solve_ivp(rhs, t_span, y0, t_eval=t_eval, **kwargs)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 3.3-A — First-order system

A system of $n$ first-order equations has the vector form

$$
\mathbf{x}'=\mathbf{f}(t,\mathbf{x}),
\qquad
\mathbf{x}(t_0)=\mathbf{x}_0.
$$

It is linear when

$$
\mathbf{x}'=A(t)\mathbf{x}+\mathbf{g}(t).
$$

Otherwise it is nonlinear.

### Principle 3.3-B — Compartment balance

For each compartment,

$$
\text{rate of change}
=
\text{total inflow}
-
\text{total outflow}
+
\text{internal production}
-
\text{internal loss}.
$$

Flows between compartments appear with opposite signs in the two corresponding equations.

### Proposition 3.3-C — Positivity

For many compartment and population models, if all initial amounts are nonnegative and every outward rate vanishes when its source compartment is empty, then the nonnegative region is forward invariant.

### Model 3.3-D — Radioactive decay chain

For a chain $X_1\to X_2\to X_3$ with decay constants $k_1,k_2>0$ and stable $X_3$,

$$
X_1'=-k_1X_1,
$$

$$
X_2'=k_1X_1-k_2X_2,
$$

$$
X_3'=k_2X_2.
$$

The total amount $X_1+X_2+X_3$ is conserved.

### Model 3.3-E — Lotka–Volterra predator–prey equations

With prey $x(t)$ and predator $y(t)$,

$$
x'=\alpha x-\beta xy,
$$

$$
y'=\delta xy-\gamma y,
$$

where all parameters are positive.

The equilibria are

$$
(0,0)
\qquad\text{and}\qquad
\left(\frac{\gamma}{\delta},\frac{\alpha}{\beta}\right).
$$

### Theorem 3.3-F — Lotka–Volterra first integral

In the positive quadrant,

$$
H(x,y)
=
\delta x-\gamma\ln x+\beta y-\alpha\ln y
$$

is constant along non-equilibrium trajectories of the ideal Lotka–Volterra system.

### Model 3.3-G — Logistic competition

A common competition model is

$$
x'=r_1x\left(1-\frac{x+\alpha_{12}y}{K_1}\right),
$$

$$
y'=r_2y\left(1-\frac{y+\alpha_{21}x}{K_2}\right).
$$

The nullclines determine whether one species excludes the other, both coexist, or the outcome depends on initial conditions.

### Classroom Checkpoint — Conservation in Compartments

Two closed tanks exchange material, with no external inflow or outflow. What should the derivative of the total amount satisfy?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Linear versus nonlinear systems

Examples:

$$
\begin{aligned}
x'&=-2x+y,\\
y'&=3x-4y
\end{aligned}
$$

is linear, while

$$
\begin{aligned}
x'&=x-xy,\\
y'&=-y+xy
\end{aligned}
$$

is nonlinear because of the product $xy$.

## 2. Radioactive decay chain

Suppose a parent isotope decays into a radioactive daughter, which then decays into a stable product.

The daughter amount can initially rise because it is being produced faster than it is lost.

In [ ]:
def decay_chain(k1=0.12, k2=0.35, X10=100, X20=0, X30=0, final_time=60):
    def rhs(t, X):
        x1, x2, x3 = X
        return [-k1*x1, k1*x1-k2*x2, k2*x2]

    sol = safe_solve(rhs, (0, final_time), [X10, X20, X30], rtol=1e-9, atol=1e-11)

    plt.plot(sol.t, sol.y[0], label="parent")
    plt.plot(sol.t, sol.y[1], label="daughter")
    plt.plot(sol.t, sol.y[2], label="stable product")
    plt.xlabel("time")
    plt.ylabel("amount")
    plt.title("Radioactive decay chain")
    plt.legend()
    plt.show()

    total = sol.y.sum(axis=0)
    print("Maximum total-amount drift:", np.max(np.abs(total-total[0])))

if WIDGETS_AVAILABLE:
    interact(
        decay_chain,
        k1=FloatSlider(min=0.02, max=0.5, step=0.02, value=0.12),
        k2=FloatSlider(min=0.02, max=0.8, step=0.02, value=0.35),
        X10=FloatSlider(min=10, max=200, step=10, value=100),
        X20=FloatSlider(min=0, max=100, step=5, value=0),
        X30=FloatSlider(min=0, max=100, step=5, value=0),
        final_time=IntSlider(min=10, max=150, step=10, value=60)
    )
else:
    decay_chain()

## 3. Two connected mixing tanks

Let $x_1(t)$ and $x_2(t)$ be the solute amounts in two constant-volume tanks.

Each flow contributes:

- a negative term to the source tank,
- a positive term to the destination tank.

In [ ]:
# Original schematic for two connected tanks
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axis("off")

tank1 = Rectangle((0.15, 0.2), 0.25, 0.5, fill=False, linewidth=2)
tank2 = Rectangle((0.60, 0.2), 0.25, 0.5, fill=False, linewidth=2)
ax.add_patch(tank1)
ax.add_patch(tank2)

ax.text(0.275, 0.45, r"Tank 1" + "\n" + r"$x_1(t),V_1$", ha="center", va="center")
ax.text(0.725, 0.45, r"Tank 2" + "\n" + r"$x_2(t),V_2$", ha="center", va="center")

ax.add_patch(FancyArrowPatch((0.40, 0.58), (0.60, 0.58), arrowstyle="->", mutation_scale=18))
ax.add_patch(FancyArrowPatch((0.60, 0.32), (0.40, 0.32), arrowstyle="->", mutation_scale=18))
ax.text(0.50, 0.66, r"$q_{12}$", ha="center")
ax.text(0.50, 0.22, r"$q_{21}$", ha="center")
plt.show()

For equal volumes $V$ and exchange rates $q_{12}$ and $q_{21}$,

$$
x_1'
=
-q_{12}\frac{x_1}{V}
+
q_{21}\frac{x_2}{V},
$$

$$
x_2'
=
q_{12}\frac{x_1}{V}
-
q_{21}\frac{x_2}{V}.
$$

The total salt $x_1+x_2$ is conserved if there is no external inflow or outflow.

In [ ]:
def two_tanks(V=100, q12=6, q21=3, x10=80, x20=10, final_time=120):
    def rhs(t, x):
        x1, x2 = x
        return [-q12*x1/V+q21*x2/V,
                q12*x1/V-q21*x2/V]

    sol = safe_solve(rhs, (0, final_time), [x10, x20], rtol=1e-9, atol=1e-11)

    plt.plot(sol.t, sol.y[0], label="tank 1")
    plt.plot(sol.t, sol.y[1], label="tank 2")
    plt.axhline((x10+x20)*q21/(q12+q21), linestyle=":")
    plt.axhline((x10+x20)*q12/(q12+q21), linestyle=":")
    plt.xlabel("time")
    plt.ylabel("solute amount")
    plt.title("Exchange between two tanks")
    plt.legend()
    plt.show()

    total = sol.y[0]+sol.y[1]
    print("Maximum conservation error:", np.max(np.abs(total-total[0])))

if WIDGETS_AVAILABLE:
    interact(
        two_tanks,
        V=FloatSlider(min=40, max=200, step=10, value=100),
        q12=FloatSlider(min=1, max=15, step=1, value=6),
        q21=FloatSlider(min=1, max=15, step=1, value=3),
        x10=FloatSlider(min=0, max=200, step=10, value=80),
        x20=FloatSlider(min=0, max=200, step=10, value=10),
        final_time=IntSlider(min=20, max=300, step=20, value=120)
    )
else:
    two_tanks()

## 4. Predator–prey modeling assumptions

For prey $x$ and predators $y$:

- without predators, prey grow exponentially at rate $\alpha x$;
- without prey, predators decline at rate $\gamma y$;
- encounters occur at a rate proportional to $xy$;
- encounters decrease prey by $\beta xy$;
- encounters support predator growth by $\delta xy$.

This produces

$$
x'=\alpha x-\beta xy,
$$

$$
y'=\delta xy-\gamma y.
$$

In [ ]:
def predator_prey(alpha=1.0, beta=0.1, delta=0.075, gamma=1.5,
                  x0=20, y0=8, final_time=40):
    def rhs(t, z):
        x, y = z
        return [alpha*x-beta*x*y,
                delta*x*y-gamma*y]

    sol = safe_solve(rhs, (0, final_time), [x0, y0], rtol=1e-9, atol=1e-11)

    plt.plot(sol.t, sol.y[0], label="prey")
    plt.plot(sol.t, sol.y[1], label="predator")
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Predator and prey time series")
    plt.legend()
    plt.show()

    plt.plot(sol.y[0], sol.y[1], linewidth=2)
    plt.scatter([x0], [y0], s=70, label="initial state")
    plt.scatter([gamma/delta], [alpha/beta], s=70, label="positive equilibrium")
    plt.xlabel("prey")
    plt.ylabel("predator")
    plt.title("Phase-plane trajectory")
    plt.legend()
    plt.show()

    H = delta*sol.y[0]-gamma*np.log(sol.y[0]) + beta*sol.y[1]-alpha*np.log(sol.y[1])
    print("Maximum first-integral drift:", np.max(np.abs(H-H[0])))

if WIDGETS_AVAILABLE:
    interact(
        predator_prey,
        alpha=FloatSlider(min=0.2, max=2.0, step=0.1, value=1.0),
        beta=FloatSlider(min=0.02, max=0.3, step=0.01, value=0.1),
        delta=FloatSlider(min=0.02, max=0.2, step=0.005, value=0.075),
        gamma=FloatSlider(min=0.2, max=3.0, step=0.1, value=1.5),
        x0=FloatSlider(min=2, max=60, step=1, value=20),
        y0=FloatSlider(min=2, max=40, step=1, value=8),
        final_time=IntSlider(min=10, max=100, step=5, value=40)
    )
else:
    predator_prey()

### Phase lag

Typically the prey population peaks first. The predator population peaks later because predator growth responds to prey abundance.

In [ ]:
alpha, beta, delta, gamma = 1.0, 0.1, 0.075, 1.5
def rhs(t, z):
    x, y = z
    return [alpha*x-beta*x*y, delta*x*y-gamma*y]

sol = safe_solve(rhs, (0, 50), [20, 8], rtol=1e-10, atol=1e-12)
prey_peak_idx = np.argmax(sol.y[0])
pred_peak_idx = np.argmax(sol.y[1])

plt.plot(sol.t, sol.y[0], label="prey")
plt.plot(sol.t, sol.y[1], label="predator")
plt.axvline(sol.t[prey_peak_idx], linestyle="--", label="prey peak")
plt.axvline(sol.t[pred_peak_idx], linestyle=":", label="predator peak")
plt.legend()
plt.show()

print("First displayed prey peak time:", sol.t[prey_peak_idx])
print("First displayed predator peak time:", sol.t[pred_peak_idx])

## 5. Logistic competition

For

$$
x'=r_1x\left(1-\frac{x+\alpha_{12}y}{K_1}\right),
$$

$$
y'=r_2y\left(1-\frac{y+\alpha_{21}x}{K_2}\right),
$$

the nonzero nullclines are

$$
x+\alpha_{12}y=K_1,
$$

$$
\alpha_{21}x+y=K_2.
$$

Their positions determine the long-term outcome.

In [ ]:
def competition_explorer(r1=0.6, r2=0.5, K1=100, K2=100,
                         a12=0.7, a21=0.6, x0=20, y0=25, final_time=80):
    def rhs(t, z):
        x, y = z
        return [
            r1*x*(1-(x+a12*y)/K1),
            r2*y*(1-(y+a21*x)/K2)
        ]

    sol = safe_solve(rhs, (0, final_time), [x0, y0], rtol=1e-9, atol=1e-11)

    plt.plot(sol.t, sol.y[0], label="species x")
    plt.plot(sol.t, sol.y[1], label="species y")
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Competition dynamics")
    plt.legend()
    plt.show()

    x_grid = np.linspace(0, max(K1, K2/a21)*1.15, 300)
    y_null_1 = (K1-x_grid)/a12
    y_null_2 = K2-a21*x_grid

    plt.plot(x_grid, y_null_1, label="x'=0 nullcline")
    plt.plot(x_grid, y_null_2, linestyle="--", label="y'=0 nullcline")
    plt.plot(sol.y[0], sol.y[1], linewidth=2, label="trajectory")
    plt.xlim(0, max(K1, K2/a21)*1.1)
    plt.ylim(0, max(K2, K1/a12)*1.1)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Competition phase plane")
    plt.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        competition_explorer,
        r1=FloatSlider(min=0.1, max=1.2, step=0.1, value=0.6),
        r2=FloatSlider(min=0.1, max=1.2, step=0.1, value=0.5),
        K1=FloatSlider(min=40, max=200, step=10, value=100),
        K2=FloatSlider(min=40, max=200, step=10, value=100),
        a12=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.7),
        a21=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.6),
        x0=FloatSlider(min=2, max=150, step=2, value=20),
        y0=FloatSlider(min=2, max=150, step=2, value=25),
        final_time=IntSlider(min=20, max=150, step=10, value=80)
    )
else:
    competition_explorer()

## Optional extension — Electrical networks

A multi-loop electrical network produces a linear system by combining:

- Kirchhoff's current law at branch points;
- Kirchhoff's voltage law around each loop;
- constitutive laws such as $V_R=Ri$ and $V_L=L\,di/dt$.

The main modeling skill is to choose independent currents and maintain consistent signs.

## Classroom Checkpoint — Exit Check

For the predator–prey system

$$
x'=1.2x-0.08xy,
$$

$$
y'=0.04xy-0.8y,
$$

find the positive equilibrium.

> Pause here. Let students commit to an answer before running the next cell.